# Case2 CER ML — estimate it: predictive maintenance

Scaffold for Case 2: OLS and gradient-boosted cost estimating relationships, cross-validation, SHAP, and a life-cycle roll-up.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ri128-rutgers/quality-management-fall2026/blob/main/case2_cer_ml.ipynb) · Also opens in Azure ML Studio notebooks or any Jupyter; nothing here needs a GPU.

**How to use:** run every cell top to bottom once. Cells marked `# >>> YOUR WORK` are where you change inputs or add interpretation. Keep your AI-assistant transcript if you used one; the verification log asks what you checked.

In [ ]:
# Setup — works in Colab and locally
import os, sys, io, urllib.request
REPO = 'ri128-rutgers/quality-management-fall2026'
DATA_BASE = f'https://raw.githubusercontent.com/{REPO}/main/' if 'google.colab' in sys.modules or not os.path.exists('../data') else '../data/'
if 'google.colab' in sys.modules:
    urllib.request.urlretrieve(f'https://raw.githubusercontent.com/{REPO}/main/tvm.py', 'tvm.py')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from tvm import *
pd.options.display.float_format = '{:,.2f}'.format
def load(name): return pd.read_csv(DATA_BASE + name)
print('data from', DATA_BASE)

In [ ]:
progs = load('case2_programs.csv'); A = load('case2_concept_assumptions.csv').set_index('parameter')['value']
print(progs.describe().round(1).T.head(12))

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_predict
feat = ['sensors', 'enterprise_integrations', 'ml_models', 'data_readiness_1to5']
X = progs[feat].copy(); X['in_house'] = (progs.vendor_type=='in-house').astype(int); X['platform'] = (progs.vendor_type=='platform').astype(int)
y = progs['dev_integration_person_months']
# OLS on logs (classic CER form)
Xl = X.copy(); Xl[['sensors','enterprise_integrations','ml_models']] = np.log(Xl[['sensors','enterprise_integrations','ml_models']])
ols = LinearRegression().fit(Xl, np.log(y))
cv = KFold(5, shuffle=True, random_state=0)
pred_ols = np.exp(cross_val_predict(LinearRegression(), Xl, np.log(y), cv=cv))
gbm = GradientBoostingRegressor(n_estimators=300, max_depth=2, learning_rate=0.05, random_state=0)
pred_gbm = cross_val_predict(gbm, X, y, cv=cv)
rmse = lambda p: np.sqrt(np.mean((p-y)**2))
print('cross-validated RMSE (person-months): OLS', round(rmse(pred_ols),1), ' GBM', round(rmse(pred_gbm),1))
print('OLS coefficients:', dict(zip(Xl.columns, ols.coef_.round(3))), 'R2 in-sample', round(ols.score(Xl, np.log(y)),3))

In [ ]:
# SHAP for the boosted model (explain it to a reviewer)
import shap
gbm.fit(X, y)
exp = shap.TreeExplainer(gbm); sv = exp.shap_values(X)
shap.summary_plot(sv, X, show=False); plt.show()

In [ ]:
# Estimate the new program with both models; is it in the relevant range?
new = pd.DataFrame([{'sensors': float(A['new_program_sensors_planned']), 'enterprise_integrations': float(A['new_program_enterprise_integrations']), 'ml_models': float(A['new_program_ml_models']), 'data_readiness_1to5': float(A['new_program_data_readiness_1to5']), 'in_house': 0, 'platform': 1}])
nl = new.copy(); nl[['sensors','enterprise_integrations','ml_models']] = np.log(nl[['sensors','enterprise_integrations','ml_models']])
print('OLS estimate PM:', round(float(np.exp(ols.predict(nl))[0]),1), ' GBM estimate PM:', round(float(gbm.predict(new)[0]),1))
print('relevant range, sensors:', progs.sensors.min(), '-', progs.sensors.max())
# >>> YOUR WORK: pick a model, state why, and convert PM to dollars at the loaded rate in A

In [ ]:
# Life-cycle roll-up (10 years, constant dollars then discounted at the real MARR). Replace the placeholders.
N = int(A['horizon_years']); r = float(A['MARR_real'])
pm_est = 110.0    # >>> YOUR WORK: from the model you chose
dev = pm_est * float(A['loaded_person_month_usd_2026'])
hw = float(A['new_program_sensors_planned'])*float(A['sensor_unit_cost_usd']) + float(A['new_program_lines'])*float(A['edge_gateway_cost_per_line_usd'])
cloud = float(A['new_program_sensors_planned']) * 31 * 12   # $/sensor/month from the consortium mean; >>> YOUR WORK: justify or replace
sustain = 0.16 * pm_est * float(A['loaded_person_month_usd_2026'])
retrain = 2 * 18_000    # two retrains a year; >>> YOUR WORK: source
refresh_yr = int(A['accelerator_or_edge_refresh_years'])
cf = [-(dev+hw)] + [-(cloud+sustain+retrain) - (hw*0.35 if y % refresh_yr == 0 else 0) for y in range(1, N+1)]
cum_os = np.cumsum([-c for c in cf[1:]])
print('total constant $', round(-sum(cf)), ' discounted', round(-pw(cf, r)))
print('O&S share of total:', round(cum_os[-1]/(-sum(cf)),2), ' year cumulative O&S passes acquisition:', int(np.argmax(cum_os > dev+hw))+1)